# Разработка ML-приложения: Классификация фильмов (FastAPI + Streamlit)


В этом блокноте разрабатывается бэкенд на FastAPI и веб-интерфейс на Streamlit для классификации фильмов по их описанию на основе предобученной модели Random Forest.


## 1. Установка библиотек
Убедитесь, что все библиотеки установлены в вашем виртуальном окружении:
```bash
pip install fastapi uvicorn streamlit requests scikit-learn pymorphy3 nltk
```


## 2. Создание файла бэкенда (FastAPI)
Выполните ячейку ниже для создания файла `app_api.py`.


In [ ]:
%%writefile app_api.py
import pickle
import string
import re
import warnings
warnings.filterwarnings('ignore')

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pymorphy3

# Инициализация nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

app = FastAPI(title="Movie Classifier API", description="FastAPI для классификации описаний фильмов")

# Список стоп-слов
russian_stopwords = stopwords.words("russian")
russian_stopwords.extend([
    'т.д.', 'т', 'д', 'это','который','свой','своём','всем','всё','её','оба','ещё','должный',
    "фильм", "кино", "режиссер", "актер", "роль", "герой", "жизнь", "человек",
    "один", "два", "три", "новый", "своем", "всем", "всё", "время", "история"
])

morph = pymorphy3.MorphAnalyzer()

def clean_and_preprocess(text):
    # Очистка
    text = str(text).lower()
    text = ''.join([ch for ch in text if ch not in string.punctuation])
    text = ''.join([i if not i.isdigit() else '' for i in text])
    text = ''.join([i if i.isalpha() else ' ' for i in text])
    text = re.sub(r'\s+', ' ', text, flags=re.I)
    text = re.sub('[a-z]', '', text, flags=re.I)
    st = '❯\xa0—«»'
    text = ''.join([ch if ch not in st else ' ' for ch in text])
    
    # Лемматизация
    tokens = word_tokenize(text)
    res = list()
    for word in tokens:
        p = morph.parse(word)[0]
        res.append(p.normal_form)
    
    # Токенизация и стоп-слова
    final_tokens = [token for token in res if token not in russian_stopwords and len(token) > 2]
    return " ".join(final_tokens)

# Загрузка предобученной модели и векторизатора
try:
    with open('model_rf.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    print("Модель и векторизатор успешно загружены!")
except Exception as e:
    print(f"Ошибка загрузки модели: {e}")

class TextRequest(BaseModel):
    text: str

@app.post("/predict")
def predict_topic(request: TextRequest):
    if not request.text.strip():
        raise HTTPException(status_code=400, detail="Текст запроса пуст")
    
    try:
        clean_txt = clean_and_preprocess(request.text)
        vectorized = vectorizer.transform([clean_txt])
        prediction = model.predict(vectorized)[0]
        probabilities = model.predict_proba(vectorized)[0]
        
        return {
            "predicted_class": int(prediction),
            "probabilities": [float(p) for p in probabilities]
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)



## 3. Создание файла фронтенда (Streamlit)
Выполните ячейку ниже для создания файла `app_streamlit.py`.


In [ ]:
%%writefile app_streamlit.py
import streamlit as st
import requests
import numpy as np
import pandas as pd

st.set_page_config(page_title="Movie Classifier", page_icon="🎬", layout="centered")

st.title("🎬 Классификатор фильмов")
st.write("Введите описание фильма, и наша модель определит его тему!")

# Карта тем
topic_names = {
    0: '0 - Герой и судьба человека',
    1: '1 – Детективы и расследования',
    2: '2 – Военная драма / История',
    3: '3 – Научная фантастика / Время',
    4: '4 – Семейные отношения / Драма',
    5: '5 – Любовь и романтика',
    6: '6 – Приключения / Пираты'
}

text_input = st.text_area("Описание фильма:", height=150, placeholder="Например: Группа исследователей отправляется в путешествие во времени...")

if st.button("Определить категорию"):
    if text_input.strip():
        with st.spinner("Классификация текста..."):
            try:
                response = requests.post(
                    "http://127.0.0.1:8000/predict", 
                    json={"text": text_input}
                )
                if response.status_code == 200:
                    result = response.json()
                    pred_class = result["predicted_class"]
                    probs = result["probabilities"]
                    
                    st.success(f"**Предсказанный класс:** {topic_names.get(pred_class, f'Тема {pred_class}')}")
                    
                    # Построение графика вероятностей
                    prob_df = pd.DataFrame({
                        'Тема': [topic_names.get(i, f'Тема {i}') for i in range(len(probs))],
                        'Вероятность': probs
                    })
                    st.write("### Распределение вероятностей:")
                    st.bar_chart(prob_df.set_index('Тема'))
                else:
                    st.error(f"Ошибка сервера: {response.json().get('detail', 'Неизвестная ошибка')}")
            except Exception as e:
                st.error(f"Не удалось подключиться к FastAPI бэкенду. Убедитесь, что он запущен на http://127.0.0.1:8000. Ошибка: {e}")
    else:
        st.warning("Пожалуйста, введите текст описания.")



## 4. Инструкция по запуску приложений
Чтобы запустить классификатор фильмов, откройте два терминала:

1. В **первом терминале** запустите бэкенд FastAPI:
```bash
uvicorn app_api:app --reload --port 8000
```
2. Во **втором терминале** запустите интерфейс Streamlit:
```bash
streamlit run app_streamlit.py --server.port 8501
```
Откройте в браузере адрес `http://localhost:8501` для работы с приложением.
